# Ridge Regression Baseline on Saved GBN Experiments

This notebook reruns **only the Ridge regression baseline** on the already-saved synthetic GBN experiments.

It reuses the saved:
- `true_gbn.json`
- `train.csv`
- `test.csv`

so that Ridge is evaluated on the **same graph structures and same datasets** used previously for the MLE/OLS and Neural Network experiments.

The notebook:

1. finds all saved experiment folders,
2. loads the true GBN parameters and saved train/test splits,
3. fits a node-wise Ridge regression CPD for each node,
4. computes KL divergence against the true conditional Gaussian distribution,
5. records **runtime per experiment**,
6. saves both **experiment-level** and **grouped summary** CSV files.

In [1]:
# Core imports
import json
import math
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold

## Configuration

Set the base results directory to the folder that contains your saved synthetic experiment folders.

The notebook saves:
- one experiment-level CSV with one row per saved run,
- one grouped summary CSV aggregated by graph size / in-degree / sample size.

In [2]:
# =========================
# User configuration
# =========================

# Folder containing all saved experiments
BASE_RESULTS_DIR = Path("21-01_Test_Final_Results")   # <-- change if needed

# Output folder
OUTPUT_DIR = BASE_RESULTS_DIR / "ridge_only_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ridge settings
USE_RIDGE_CV = True
RIDGE_ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0]
RIDGE_CV_FOLDS = 3

# Numerical safeguard for residual std
MIN_STD = 1e-3

print("BASE_RESULTS_DIR:", BASE_RESULTS_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

BASE_RESULTS_DIR: C:\Users\Aashna\Documents\Masters\Final Experiment 1\21-01_Test_Final_Results\21-01_Test_Final_Results
OUTPUT_DIR: C:\Users\Aashna\Documents\Masters\Final Experiment 1\21-01_Test_Final_Results\21-01_Test_Final_Results\ridge_only_results


## Helper functions

In [3]:
def load_true_gbn_json(json_path):
    with open(json_path, "r") as f:
        gbn_info = json.load(f)

    nodes = gbn_info["nodes"]
    parents_dict = gbn_info["parents"]
    mu_true = [float(x) for x in gbn_info["mu"]]
    sigma_true = [float(x) for x in gbn_info["sigma"]]
    weights_true = [np.array(w, dtype=float) for w in gbn_info["weights"]]

    return nodes, parents_dict, mu_true, weights_true, sigma_true, gbn_info


def find_true_gbn_json_for_experiment(exp_dir):
    exp_dir = Path(exp_dir)
    for parent in [exp_dir] + list(exp_dir.parents):
        candidate = parent / "true_gbn.json"
        if candidate.exists():
            return candidate
    return None


def parse_first_int(pattern, text):
    m = re.search(pattern, str(text))
    return int(m.group(1)) if m else np.nan


def extract_metadata_from_paths(exp_dir, gbn_json_info, parents_dict):
    exp_dir = Path(exp_dir)
    path_str = str(exp_dir)

    graph_size = parse_first_int(r"GBN_(\d+)", path_str)
    declared_indegree = parse_first_int(r"indegree_(\d+)", path_str)
    sample_size = parse_first_int(r"samples_(\d+)", path_str)

    simulation_id = np.nan
    for part in exp_dir.parts:
        m = re.match(r"(\d+)\.\s*GBN_", part)
        if m:
            simulation_id = int(m.group(1))
            break

    parent_counts = [len(parents_dict[n]) for n in gbn_json_info["nodes"]]
    actual_max_parents = max(parent_counts) if parent_counts else 0
    mean_parent_count = float(np.mean(parent_counts)) if parent_counts else 0.0
    n_root_nodes = int(sum(pc == 0 for pc in parent_counts))

    return {
        "graph_size": graph_size,
        "declared_indegree": declared_indegree,
        "actual_max_parents": actual_max_parents,
        "mean_parent_count": mean_parent_count,
        "n_root_nodes": n_root_nodes,
        "sample_size": sample_size,
        "simulation_id": simulation_id,
        "n_nodes_in_json": len(gbn_json_info["nodes"]),
    }

## KL utilities

Ridge remains a **linear Gaussian** baseline, so it can be evaluated using the same conditional Gaussian KL structure as your existing linear baseline.

In [4]:
def baseline_conditional_gaussian_kl_np(
    true_mu,
    true_weights,
    true_sigma,
    base_mu,
    base_weights,
    base_sigma,
    x
):
    true_mu = float(true_mu)
    true_sigma = float(true_sigma)
    base_mu = float(base_mu)
    base_sigma = float(base_sigma)

    true_weights = np.asarray(true_weights, dtype=float)
    base_weights = np.asarray(base_weights, dtype=float)
    x = np.asarray(x, dtype=float)

    mu_true_cond = true_mu + (x @ true_weights if true_weights.size > 0 else 0.0)
    mu_base_cond = base_mu + (x @ base_weights if base_weights.size > 0 else 0.0)

    var_true = true_sigma ** 2
    var_base = base_sigma ** 2

    kl = 0.5 * (
        np.log(var_base / var_true)
        + (var_true + (mu_true_cond - mu_base_cond) ** 2) / var_base
        - 1.0
    )
    return float(kl)


def compute_baseline_lgbn_kl(
    data,
    mu_true,
    weights_true,
    sigma_true,
    mu_pred,
    weights_pred,
    sigma_pred,
    parents
):
    node_names = list(data.columns)
    node_to_idx = {node: i for i, node in enumerate(node_names)}
    data_np = data.values.astype(float)

    kl_node_list = []

    for i, node in enumerate(node_names):
        parent_names = parents[node]
        parent_indices = [node_to_idx[p] for p in parent_names]

        if len(parent_indices) > 0:
            x = data_np[:, parent_indices]
        else:
            x = np.zeros((len(data_np), 0), dtype=float)

        kl_values = [
            baseline_conditional_gaussian_kl_np(
                true_mu=mu_true[i],
                true_weights=weights_true[i],
                true_sigma=sigma_true[i],
                base_mu=mu_pred[i],
                base_weights=weights_pred[i],
                base_sigma=sigma_pred[i],
                x=x_row
            )
            for x_row in x
        ]

        kl_node_list.append(float(np.mean(kl_values)))

    return float(np.mean(kl_node_list)), kl_node_list

## Ridge baseline training

In [5]:
def fit_ridge_for_node(X, y, use_cv=True, alpha_grid=None, cv_folds=3):
    if X.shape[1] == 0:
        mu = float(np.mean(y))
        resid = y - mu
        sigma = max(float(np.std(resid, ddof=1)), MIN_STD)
        return mu, np.array([], dtype=float), sigma, np.nan

    if use_cv:
        ridge = Ridge(fit_intercept=True)
        cv = KFold(n_splits=cv_folds, shuffle=True, random_state=42)
        grid = GridSearchCV(
            estimator=ridge,
            param_grid={"alpha": alpha_grid},
            scoring="neg_mean_squared_error",
            cv=cv,
            n_jobs=-1
        )
        grid.fit(X, y)
        model = grid.best_estimator_
        alpha_used = float(grid.best_params_["alpha"])
    else:
        model = Ridge(alpha=1.0, fit_intercept=True)
        model.fit(X, y)
        alpha_used = float(model.alpha)

    y_hat = model.predict(X)
    resid = y - y_hat
    sigma = max(float(np.std(resid, ddof=1)), MIN_STD)

    return float(model.intercept_), np.array(model.coef_, dtype=float), sigma, alpha_used


def train_ridge_baseline(train_data, nodes, parents_dict, use_cv=True, alpha_grid=None, cv_folds=3):
    mu_pred = []
    weights_pred = []
    sigma_pred = []
    alpha_used = []

    for node in nodes:
        parent_names = parents_dict[node]
        y = train_data[node].values.astype(float)
        X = train_data[parent_names].values.astype(float) if len(parent_names) > 0 else np.zeros((len(y), 0))

        mu, w, sigma, alpha = fit_ridge_for_node(
            X=X,
            y=y,
            use_cv=use_cv,
            alpha_grid=alpha_grid,
            cv_folds=cv_folds
        )

        mu_pred.append(mu)
        weights_pred.append(w)
        sigma_pred.append(sigma)
        alpha_used.append(alpha)

    return mu_pred, weights_pred, sigma_pred, alpha_used

## Discover saved experiments

In [6]:
def discover_saved_experiments(base_dir):
    base_dir = Path(base_dir)
    experiments = []

    for train_path in base_dir.rglob("train.csv"):
        exp_dir = train_path.parent
        test_path = exp_dir / "test.csv"

        if not test_path.exists():
            continue

        true_gbn_path = find_true_gbn_json_for_experiment(exp_dir)
        if true_gbn_path is None:
            continue

        experiments.append({
            "exp_dir": exp_dir,
            "train_path": train_path,
            "test_path": test_path,
            "true_gbn_path": true_gbn_path,
        })

    experiments = sorted(experiments, key=lambda x: str(x["exp_dir"]))
    return experiments


experiments = discover_saved_experiments(BASE_RESULTS_DIR)
print(f"Found {len(experiments)} saved experiments.")
if experiments:
    display(pd.DataFrame([{
        "exp_dir": str(experiments[0]["exp_dir"]),
        "true_gbn_path": str(experiments[0]["true_gbn_path"])
    }]))

Found 160 saved experiments.


,exp_dir,true_gbn_path
0,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...


## Run Ridge on the saved experiments

This records:
- KL divergence,
- node-level KL summary statistics,
- selected Ridge alpha summary,
- **runtime in seconds** per saved experiment.

In [7]:
records = []

for item in tqdm(experiments, desc="Running Ridge baseline"):
    exp_dir = item["exp_dir"]
    train_path = item["train_path"]
    test_path = item["test_path"]
    true_gbn_path = item["true_gbn_path"]

    train_data = pd.read_csv(train_path)
    test_data = pd.read_csv(test_path)

    nodes, parents_dict, mu_true, weights_true, sigma_true, gbn_json_info = load_true_gbn_json(true_gbn_path)

    # Enforce node order consistency
    train_data = train_data[nodes].copy()
    test_data = test_data[nodes].copy()

    meta = extract_metadata_from_paths(exp_dir, gbn_json_info, parents_dict)

    ridge_start = time.perf_counter()

    mu_ridge, weights_ridge, sigma_ridge, alpha_used = train_ridge_baseline(
        train_data=train_data,
        nodes=nodes,
        parents_dict=parents_dict,
        use_cv=USE_RIDGE_CV,
        alpha_grid=RIDGE_ALPHA_GRID,
        cv_folds=RIDGE_CV_FOLDS
    )

    ridge_kl_avg, ridge_kl_nodes = compute_baseline_lgbn_kl(
        data=test_data,
        mu_true=mu_true,
        weights_true=weights_true,
        sigma_true=sigma_true,
        mu_pred=mu_ridge,
        weights_pred=weights_ridge,
        sigma_pred=sigma_ridge,
        parents=parents_dict
    )

    ridge_time_sec = float(time.perf_counter() - ridge_start)

    alpha_nonroot = [a for a in alpha_used if not pd.isna(a)]

    record = {
        **meta,
        "experiment_dir": str(exp_dir),
        "true_gbn_path": str(true_gbn_path),
        "ridge_kl_avg": float(ridge_kl_avg),
        "ridge_kl_node_mean": float(np.mean(ridge_kl_nodes)),
        "ridge_kl_node_std": float(np.std(ridge_kl_nodes, ddof=1)) if len(ridge_kl_nodes) > 1 else 0.0,
        "ridge_alpha_mean_nonroot": float(np.mean(alpha_nonroot)) if len(alpha_nonroot) > 0 else np.nan,
        "ridge_alpha_min_nonroot": float(np.min(alpha_nonroot)) if len(alpha_nonroot) > 0 else np.nan,
        "ridge_alpha_max_nonroot": float(np.max(alpha_nonroot)) if len(alpha_nonroot) > 0 else np.nan,
        "ridge_time_sec": ridge_time_sec,
        "ridge_time_min": ridge_time_sec / 60.0,
        "method": "ridge",
    }

    records.append(record)

results_df = pd.DataFrame(records)
results_df.head()

Running Ridge baseline:   0%|          | 0/160 [00:00<?, ?it/s]

,graph_size,declared_indegree,actual_max_parents,mean_parent_count,n_root_nodes,sample_size,simulation_id,n_nodes_in_json,experiment_dir,true_gbn_path,ridge_kl_avg,ridge_kl_node_mean,ridge_kl_node_std,ridge_alpha_mean_nonroot,ridge_alpha_min_nonroot,ridge_alpha_max_nonroot,ridge_time_sec,ridge_time_min,method
0,10,2,2,1.5,1,1000,1,10,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,0.090496,0.090496,0.175949,14.79,0.01,100.0,6.073056,0.101218,ridge
1,10,2,2,1.5,1,1400,1,10,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,0.094578,0.094578,0.166009,2.47,0.01,10.0,0.367404,0.006123,ridge
2,10,2,2,1.5,1,200,1,10,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,0.113488,0.113488,0.165882,2.78,0.01,10.0,0.370665,0.006178,ridge
3,10,2,2,1.5,1,600,1,10,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,0.090504,0.090504,0.172216,1.58,0.01,10.0,0.417179,0.006953,ridge
4,10,2,2,1.6,1,1000,2,10,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,21-01_Test_Final_Results\GBN_10\GBN_10_indegre...,0.087983,0.087983,0.169198,1.56,0.01,10.0,0.406769,0.006779,ridge


In [11]:
print(results_df.size)

3040


## Save experiment-level results

In [8]:
results_csv = OUTPUT_DIR / "ridge_experiment_level_results.csv"
results_df.to_csv(results_csv, index=False)

print(f"Saved: {results_csv}")
print(f"Rows: {len(results_df)}")

Saved: 21-01_Test_Final_Results\ridge_only_results\ridge_experiment_level_results.csv
Rows: 160


## Grouped summary

This aggregates results by:
- graph size,
- declared in-degree,
- actual maximum parents,
- sample size.

In [9]:
group_cols = ["graph_size", "declared_indegree", "actual_max_parents", "sample_size"]

summary_df = (
    results_df
    .groupby(group_cols, dropna=False)
    .agg(
        n_runs=("ridge_kl_avg", "size"),
        ridge_kl_mean=("ridge_kl_avg", "mean"),
        ridge_kl_std=("ridge_kl_avg", "std"),
        ridge_time_mean_sec=("ridge_time_sec", "mean"),
        ridge_time_std_sec=("ridge_time_sec", "std"),
        ridge_time_mean_min=("ridge_time_min", "mean"),
        ridge_alpha_mean=("ridge_alpha_mean_nonroot", "mean"),
    )
    .reset_index()
    .sort_values(group_cols)
)

summary_csv = OUTPUT_DIR / "ridge_grouped_summary_results.csv"
summary_df.to_csv(summary_csv, index=False)

print(f"Saved: {summary_csv}")
summary_df.head(20)

Saved: 21-01_Test_Final_Results\ridge_only_results\ridge_grouped_summary_results.csv


,graph_size,declared_indegree,actual_max_parents,sample_size,n_runs,ridge_kl_mean,ridge_kl_std,ridge_time_mean_sec,ridge_time_std_sec,ridge_time_mean_min,ridge_alpha_mean
0,10,2,2,200,5,0.114732,0.044932,0.321449,0.044600,0.005357,9.566000
1,10,2,2,600,5,0.102427,0.041024,0.306809,0.061818,0.005113,6.254000
2,10,2,2,1000,5,0.103792,0.040415,1.470480,2.573412,0.024508,11.432000
3,10,2,2,1400,5,0.100951,0.039279,0.326272,0.040865,0.005438,5.656000
4,10,5,3,200,2,0.239705,0.182160,0.334516,0.075107,0.005575,2.125000
5,10,5,3,600,2,0.225064,0.179146,0.320760,0.019909,0.005346,2.115000
6,10,5,3,1000,2,0.215306,0.172830,0.282464,0.018691,0.004708,2.080000
7,10,5,3,1400,2,0.220821,0.183235,0.306524,0.024847,0.005109,1.530000
8,10,5,4,200,3,0.191351,0.025117,0.287867,0.028949,0.004798,13.683333
9,10,5,4,600,3,0.157076,0.024657,0.325530,0.032664,0.005425,9.350000


## Optional merge with your existing experiment-level results

If you already have a CSV with the earlier MLE / NN experiment-level results, load it here and merge on the common identifying columns.

In [10]:
# Example:
# existing_results = pd.read_csv("PATH_TO_EXISTING_RESULTS.csv")
#
# merged = existing_results.merge(
#     results_df,
#     on=["graph_size", "declared_indegree", "sample_size", "simulation_id"],
#     how="left"
# )
#
# merged.to_csv(OUTPUT_DIR / "merged_with_existing_results.csv", index=False)
# merged.head()

## Notes

- Ridge is rerun on the **saved experiments only**.
- No new GBNs are generated.
- No new datasets are sampled.
- `ridge_time_sec` and `ridge_time_min` can be reported alongside your NN training time summaries.
- If you want results that align even more closely with your existing naming conventions, you can rename the output CSV columns after comparing them with your earlier results file.